# Tutorial 01: The Limits of Naive RAG (And Why It Breaks)

Welcome to the first notebook in the series. Before we build the robust, self-correcting CRAG (Corrective Retrieval-Augmented Generation) architecture used in this repository, we need to understand exactly *why* we are building it.

In this notebook, we will construct a "Naive RAG" pipeline—the standard LangChain tutorial approach. It will take us about 20 lines of code. We will see it work perfectly on a simple query, and then we will intentionally break it to demonstrate the hallucination and retrieval bottlenecks that occur in production environments.

### Prerequisites
* Basic understanding of Python and object-oriented programming.
* Conceptual understanding of embeddings and vector databases.
* A local running instance of Ollama (we will use `llama3` for generation and `nomic-embed-text` for embeddings).


In [ ]:
# Optional dependency setup if running outside the container sandbox:
# !pip install langchain langchain-community langchain-core faiss-cpu -q


### Step 1: The Setup & Ingestion

First, let's import our tools and create a small, in-memory knowledge base. We will use a dummy text about a system architecture (specifically, the hardware fencing concept used in this repository).


In [ ]:
import os
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Our dummy knowledge base
documents = [
    "The CRAG system utilizes Asymmetric Hardware Fencing. Generative inference is strictly fenced to the GPU.",
    "Vector storage, dense embeddings, and cross-encoder reranking are offloaded to CPU and system RAM.",
    "This asymmetric split eliminates VRAM bleed, allowing heavy enterprise workloads to run on consumer hardware like the RTX 4060.",
    "Store baristas and shift managers use the frontend Gradio UI for daily query operations, backed by a 60-day log evaluation window."
]

# 2. Initialize our local embedding model via Ollama
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 3. Create a naive, in-memory vector store
vectorstore = FAISS.from_texts(documents, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("✅ Documents ingested and vector store ready.")


### Step 2: The "Happy Path"

Now, let's build the standard generation chain. We retrieve documents based on semantic similarity, stuff them into a prompt, and ask the LLM for an answer.


In [ ]:
# 1. Initialize the local LLM
llm = Ollama(model="llama3")

# 2. Define a standard RAG prompt
template = """Answer the question based ONLY on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)

# 3. Build the LCEL (LangChain Expression Language) Chain
naive_rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test the Happy Path
query = "What hardware is generative inference fenced to?"
response = naive_rag_chain.invoke(query)

print(f"Question: {query}")
print(f"Answer: {response}")


**Expected Output:** *Generative inference is strictly fenced to the GPU.*

It works perfectly. The retriever found the exact chunk, the LLM read it, and it answered the question. This is where most beginner tutorials end.

---

### Step 3: The Breakdown (Why Naive RAG Fails in Production)

In the real world, users do not ask perfectly phrased questions that match your document chunks. They ask ambiguous questions, make typos, or try to chat conversationally.

Let's see what happens when a user simply says "Hello" to the system.


In [ ]:
# The Failure Mode
bad_query = "Hello, good morning!"

# Let's peek at what the retriever actually fetches for a greeting
retrieved_docs = retriever.invoke(bad_query)
print("--- Retrieved Context for 'Hello, good morning!' ---")
for doc in retrieved_docs:
    print(f"- {doc.page_content}")

print("\n--- LLM Response ---")
bad_response = naive_rag_chain.invoke(bad_query)
print(bad_response)


**What just happened?**

1. The embedding model converted "Hello, good morning!" into a vector array.
2. It searched the vector database for the "closest" mathematical match.
3. It pulled completely irrelevant documents about GPUs and baristas simply because they were the closest vectors in a very small database.
4. The LLM received a prompt saying: *"Answer 'Hello, good morning!' based ONLY on this context about RTX 4060s."*

Depending on the LLM's safety tuning, it will either hallucinate a bizarre connection between mornings and GPUs, or stubbornly refuse to answer a friendly greeting. Furthermore, you just wasted 2-3 seconds of compute time and token generation on a vector search for a simple greeting.

### The Solution: We Need a State Machine

Naive RAG assumes a linear pipeline: `Query -> Search -> Generate`.

To build a production system, we must abandon linear pipelines and adopt **cyclic state machines**. We need the system to "think" before it acts:

* *Is this query a greeting?* If yes, skip the database entirely and just say hello back. (Routing)
* *Are the retrieved documents actually relevant to the question?* If no, throw them away and rewrite the user's query to try again. (Grading & Correction)

This is the core concept of **Corrective RAG (CRAG)**.

---

### 🏋️ Challenge

Before moving to the next notebook, try to "break" the naive RAG chain yourself:

1. Add a new document to the `documents` list about a completely unrelated topic (e.g., a recipe for baking bread).
2. Query the system with: *"How do I eliminate VRAM bleed while baking bread?"*
3. Look at the retrieved documents and the final LLM response. Notice how the retriever forces the LLM to blend contradictory contexts.

In **Tutorial 02: Adding the Router**, we will build our first LangGraph node to intercept queries *before* they hit the database, saving compute resources and preventing the exact failure mode we saw today.
